In [9]:
!pip install -q \
    cudf-cu12 cuml-cu12 cugraph-cu12 cupy-cuda12x \
    --extra-index-url=https://pypi.nvidia.com/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 128.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 276.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 260.3 MB/s eta 0:00:00


In [10]:
%load_ext cuml.accel


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.datasets import load_wine
from sklearn.datasets import load_iris

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

from sklearn.metrics import confusion_matrix, classification_report

from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier



In [12]:
df = load_iris(as_frame=True).frame
df.sample(10)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
40,5.0,3.5,1.3,0.3,0
111,6.4,2.7,5.3,1.9,2
15,5.7,4.4,1.5,0.4,0
18,5.7,3.8,1.7,0.3,0
89,5.5,2.5,4.0,1.3,1
73,6.1,2.8,4.7,1.2,1
149,5.9,3.0,5.1,1.8,2
37,4.9,3.6,1.4,0.1,0
5,5.4,3.9,1.7,0.4,0
85,6.0,3.4,4.5,1.6,1


In [13]:
x = df.drop(['target'], axis=1)
y = df['target']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)


In [14]:
model = DecisionTreeClassifier()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [15]:
grid = {
    "criterion" : ['gini', 'entrophy'],
    "max_depth" : [1,2,3,4,5,6,7,8,None],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    cv=5
)

grid_search.fit(x_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
45 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entrophy'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, None]})

In [16]:
y_grid_pred = grid_search.predict(x_test)
print(classification_report(y_test, y_grid_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

